In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import chromadb
# Import Chroma's built-in embedding function capability
from chromadb.utils import embedding_functions

print("Imports successful! Python 3.12 environment conflicts bypassed.")

Imports successful! Python 3.12 environment conflicts bypassed.


In [9]:
print("--- Section 1: Stratified Sampling ---")

# Load your dataset (Ensure the filename matches your Task 1 output)
df = pd.read_csv('Data/processed/processed_rag_data.csv')
df = df.dropna(subset=['normalized_complaint', 'Product'])

target_sample_size = 12000
test_ratio = target_sample_size / len(df)

print(f"Original clean dataset size: {len(df):,} records.")

# Split using stratified flag
_, df_sample = train_test_split(
    df, 
    test_size=test_ratio, 
    stratify=df['Product'], 
    random_state=42
)

print(f"\nStratified Sample Size Captured: {len(df_sample):,} records.")
print("\nSampled Product Proportions:")
print(df_sample['Product'].value_counts(normalize=True))

--- Section 1: Stratified Sampling ---
Original clean dataset size: 437,795 records.

Stratified Sample Size Captured: 12,000 records.

Sampled Product Proportions:
Product
Checking or savings account                           0.320500
Credit card or prepaid card                           0.248250
Money transfer, virtual currency, or money service    0.222000
Credit card                                           0.184250
Consumer Loan                                         0.021583
Money transfers                                       0.003417
Name: proportion, dtype: float64


In [10]:
print("\n--- Section 2: Text Chunking ---")

chunk_size = 500       
chunk_overlap = 50     

def custom_sliding_chunker(text, size, overlap):
    chunks = []
    if not text or len(text) == 0:
        return chunks
    
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start += (size - overlap)
        if size <= overlap:
            break
    return chunks

chunks_list = []
metadata_list = []

print("Processing narratives into text segments...")
for _, row in df_sample.iterrows():
    complaint_text = str(row['normalized_complaint'])
    complaint_id = row['Complaint ID']
    product_category = row['Product']
    
    individual_chunks = custom_sliding_chunker(complaint_text, chunk_size, chunk_overlap)
    
    for chunk in individual_chunks:
        chunks_list.append(chunk)
        metadata_list.append({
            "complaint_id": int(complaint_id),
            "product": str(product_category)
        })

print(f"Total chunks successfully generated: {len(chunks_list):,}")


--- Section 2: Text Chunking ---
Processing narratives into text segments...
Total chunks successfully generated: 23,818


In [13]:
print("\n--- Section 3: Offline Embedding and Vector Store Indexing ---")

from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize a completely local, offline vectorizer
print("Initializing offline local TF-IDF Vectorizer...")
# We limit to 384 features to mimic the exact structural output size of MiniLM
vectorizer = TfidfVectorizer(max_features=384)

# Fit the vectorizer on your chunks to build the local vocabulary matrix
print("Fitting local text matrix structures...")
tfidf_matrix = vectorizer.fit_transform(chunks_list).toarray()

# 2. Spin up the Persistent Database connection on disk
print("Connecting to local disk-based ChromaDB storage...")
chroma_client = chromadb.PersistentClient(path="./complaints_chroma_db")

# 3. Create or fetch the collection
collection = chroma_client.get_or_create_collection(name="customer_complaints")

# 4. Pipe chunks, local vectors, and metadata into the store in safe batches
print("Indexing chunks into ChromaDB...")
batch_size = 1000

for i in range(0, len(chunks_list), batch_size):
    batch_chunks = chunks_list[i : i + batch_size]
    batch_meta = metadata_list[i : i + batch_size]
    batch_ids = [f"id_chunk_{j}" for j in range(i, i + len(batch_chunks))]
    
    # Extract the pre-computed local vector slice for this batch
    batch_embeddings = tfidf_matrix[i : i + batch_size].tolist()
    
    collection.add(
        embeddings=batch_embeddings,
        documents=batch_chunks,
        metadatas=batch_meta,
        ids=batch_ids
    )

print(f"\nVector Store built successfully! Total items saved offline: {collection.count():,}")


--- Section 3: Offline Embedding and Vector Store Indexing ---
Initializing offline local TF-IDF Vectorizer...
Fitting local text matrix structures...
Connecting to local disk-based ChromaDB storage...
Indexing chunks into ChromaDB...

Vector Store built successfully! Total items saved offline: 23,818


In [14]:
print("\n--- Section 4: Running Offline Search Verification Test ---")
test_query = "unauthorized credit card fees charges account opened identity theft"

# 1. Vectorize the test question using our offline model
query_vector = vectorizer.transform([test_query]).toarray().tolist()

# 2. Query the database collection using the calculated vector matrix coordinates
results = collection.query(
    query_embeddings=query_vector,
    n_results=2
)

print(f"\nExecuted Verification Search Query: '{test_query}'")
for idx in range(len(results['documents'][0])):
    print(f"\nSemantic Match Rank #{idx+1}")
    print(f"Source Complaint ID: {results['metadatas'][0][idx]['complaint_id']}")
    print(f"Product Domain Category: {results['metadatas'][0][idx]['product']}")
    print(f"Retrieved Segment Context: \"{results['documents'][0][idx]}\"")


--- Section 4: Running Offline Search Verification Test ---

Executed Verification Search Query: 'unauthorized credit card fees charges account opened identity theft'

Semantic Match Rank #1
Source Complaint ID: 4170802
Product Domain Category: Credit card or prepaid card
Retrieved Segment Context: "ower"

Semantic Match Rank #2
Source Complaint ID: 9603599
Product Domain Category: Money transfer, virtual currency, or money service
Retrieved Segment Context: "ey"
